# Item pipelines

> Running all pipelines for one item and inspect the changes.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import mlflow
from IPython.display import display as ipython_display
from scrapy.utils.project import get_project_settings

from notebooks.render_pipeline import pipeline_trace_viewer
from pazufa_scraper_be.pardok.dokument import DrsDokument
from pazufa_scraper_be.pardok.vorgang import GesetzVorgang
from pazufa_scraper_be.pipelines import (
    BuildPaZuFaVorgang,
    DownloadDocuments,
    ExtractText,
    HandleErrors,
    PrepareDocumentUrls,
    SummarizeText,
)
from pazufa_scraper_be.tracing.spidering import run_item
from pazufa_scraper_be.tracing.trace_pipeline import get_trace_from_mlflow_db, trace_pipeline
from pazufa_scraper_be.tracing.vorgang_helper import feed_to_gesetz_vorgang_dicts, validate_vorgang

In [ ]:
db_path = Path("./item-pipelines-nb-mlflow.db").resolve()
db_path.parent.mkdir(parents=True, exist_ok=True)

db_path

In [ ]:
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("individual nb run")

```bash
mlflow server \
  --backend-store-uri "sqlite:////absolute/path/above/to/item-pipelines-nb-mlflow.db" \
  --host 127.0.0.1 \
  --port 5000
```

Then navigate to `http://127.0.0.1:5000` in your browser and go to "Experiments > Default > Traces" and there select the latest trace.

In [ ]:
item_pipelines = {
    HandleErrors: 99,
    PrepareDocumentUrls: 100,
    DownloadDocuments: 110,
    ExtractText: 120,
    SummarizeText: 130,
    BuildPaZuFaVorgang: 999,
}

In [ ]:
for pipeline_cls in item_pipelines:
    trace_pipeline(pipeline_cls)

In [ ]:
settings = get_project_settings()
settings["ITEM_PIPELINES"] = item_pipelines
# twisted reactor not needed in notebooks, skipping the reactor though so some behavior may be different
settings["TWISTED_REACTOR_ENABLED"] = False
settings

In [ ]:
def make_test_vorgang() -> GesetzVorgang:
    """Create a dummy Vorgang."""
    dokument = DrsDokument(
        art="Drs",
        art_l="Drucksache",
        herk="Pardok",
        herk_l="Pardok-System",
        typ="Antr",
        typ_l="Antrag",
        id="doc123",
        wp=19,
        reih_nr=1,
        nr="123/2024",
        dat="10.05.2024",
        lok_url="https://example.invalid/doc123.pdf",
        titel="Entwurf eines Testgesetzes",
        urheber=["Testfraktion"],
    )

    return GesetzVorgang(
        typ="Gesetz",
        typ_l="Gesetz",
        id="vorg123",
        reih_nr=0,
        nr="V123/2024",
        ir="IR123",
        sys=[1000],
        sys_l=["Test"],
        dokumente=[dokument],
    )


test_item = make_test_vorgang()
test_item

In [ ]:
cache_dir: Path = settings.get("CACHE_DIR")
feeds = list(cache_dir.glob("**/*.xml"))
feeds

In [ ]:
feed_to_use = feeds[0]
vorgang_idx = 0

gesetz_vorgang_dicts = feed_to_gesetz_vorgang_dicts(feed_to_use)

item = validate_vorgang(gesetz_vorgang_dicts[vorgang_idx])

In [ ]:
await run_item(settings, item)

In [ ]:
trace_to_view = get_trace_from_mlflow_db()

ipython_display(pipeline_trace_viewer(trace_to_view))  # ty: ignore[invalid-argument-type]